## Importación de Librerias

In [1]:
import torch
import torch.nn.functional as F
import numpy as np

Generar un conjuno de datos desde un vector

In [2]:
# un vector (tiene que ser un tensor)
vect = torch.tensor([1,2,5], dtype=torch.float)

In [3]:
# ejempo de un numero (?) # solo vamos a obtener un numero
torch.multinomial(vect,1)

tensor([1])

La clave para entender esta función es saber que torch.multinomial NUNCA devuelve los valores contenidos dentro del tensor, sino sus ÍNDICES (las posiciones en las que se encuentran).1. ¿Cómo interpreta PyTorch el tensor [1, 2, 5]?PyTorch trata los números del tensor no como valores absolutos, sino como pesos relativos de probabilidad (es decir, cuán probable es elegir cada posición):Índice 0 (contiene el valor 1): Peso $1/8 \rightarrow$ $12.5\%$ de probabilidadÍndice 1 (contiene el valor 2): Peso $2/8 \rightarrow$ $25.0\%$ de probabilidadÍndice 2 (contiene el valor 5): Peso $5/8 \rightarrow$ $62.5\%$ de probabilidad(La suma total de los pesos es $1 + 2 + 5 = 8$).2. ¿Por qué el resultado es 0 si 0 no está en el vector?Cuando ejecutas torch.multinomial(vect, 1) y obtienes tensor([0]), el resultado 0 indica la posición cero, no el número cero.Índice obtenido: 0Valor real en esa posición: vect[0] = 1.03. Código para obtener el valor real elegidoPara extraer el valor original del tensor a partir del índice generado al azar, debes indexar el tensor original con la salida del muestreo:

In [4]:
# Vector de pesos/probabilidades
vect = torch.tensor([1, 2, 5], dtype=torch.float)

# 1. Obtenemos el índice seleccionado al azar
idx = torch.multinomial(vect, num_samples=1)

# 2. Consultamos el valor real usando ese índice
valor_elegido = vect[idx]

print(f"Índice seleccionado: {idx.item()}")
print(f"Valor real correspondie: {valor_elegido.item()}")

Índice seleccionado: 2
Valor real correspondie: 5.0


. La indexación en Python empieza en 0En PyTorch (al igual que en Python y NumPy), los elementos de un tensor se posicionan contando desde el cero:$$\text{vect} = [\underbrace{1.0}_{\text{Índice 0}}, \ \underbrace{2.0}_{\text{Índice 1}}, \ \underbrace{5.0}_{\text{Índice 2}}]$$Índice 0: Corresponde al valor 1.0 (1ª posición).Índice 1: Corresponde al valor 2.0 (2ª posición).Índice 2: Corresponde al valor 5.0 (3ª posición).Por eso, al hacer vect[2], el programa busca la tercera casilla e imprime 5.0.2. ¿Por qué el Índice 2 sale seleccionado con más frecuencia?torch.multinomial toma los valores del tensor [1, 2, 5] como pesos relativos de probabilidad. La suma total de los pesos es $1 + 2 + 5 = 8$:Índice 0 (Valor 1.0): Tiene $1$ de $8$ oportunidades $\rightarrow$ $12.5\%$ de probabilidad.Índice 1 (Valor 2.0): Tiene $2$ de $8$ oportunidades $\rightarrow$ $25.0\%$ de probabilidad.Índice 2 (Valor 5.0): Tiene $5$ de $8$ oportunidades $\rightarrow$ $62.5\%$ de probabilidad.Al tener más del $62\%$ de la masa total de probabilidad, el índice 2 (asociado al valor 5.0) será el ganador la mayoría de las veces que ejecutes la función al azar.

## Algunos Errores

In [5]:
# requiere un tensor (no un array de numpy)
# torch.multinomial([1.,2,.3],1)
# torch.multinomial(np.array([1.,2,.3]),1)

In [6]:
# por default no hay muestreo por remplazo el multinomial es sin remplazo
# torch.multinomial(vect,len(vect)+1)

In [7]:
# Solo floats
# torch.multinomial(torch.tensor([1,1,1]),1)

In [8]:
# Solo non-negative Numeros
# torch.multinomial(torch.tensor([-1,1.,1]),1)

Multinomial Transforma esos datos en una funcion de probabilidad

In [9]:
# Muestrear 10 veces de ese vector
vect[torch.multinomial(vect, 10, replacement=True)]

tensor([1., 5., 2., 5., 2., 5., 5., 1., 5., 2.])

In [10]:
# ¡10.000 muestras!
mn = torch.multinomial(vect, 10000, replacement=True)

# Obtener la distribución
vals, counts = np.unique(mn, return_counts=True)

# Imprimir los valores de salida y su frecuencia de ocurrencia
for v, c in zip(vals, counts):
  print(f'"{v}" fue muestreado {c} veces ({c*100/len(mn):.2f}%)')

"0" fue muestreado 1247 veces (12.47%)
"1" fue muestreado 2504 veces (25.04%)
"2" fue muestreado 6249 veces (62.49%)


In [11]:
# Podemos observar que las expectativas de probabilidad y las probabilidades esperadas están muy cerca

# Tratar el vector como si contuviera valores de probabilidad (escalados)

# Nuevamente con más información
for v, c, vectval in zip(vals, counts, vect):
  observedFrequency = c * 100 / len(mn)
  expectedFrequency = vectval * 100 / torch.sum(vect)

  print(
      f'"{v}" fue muestreado {c} veces. Eso es {observedFrequency:.2f}%, y la'
      f' probabilidad esperada es {expectedFrequency:.2f}%'
  )

"0" fue muestreado 1247 veces. Eso es 12.47%, y la probabilidad esperada es 12.50%
"1" fue muestreado 2504 veces. Eso es 25.04%, y la probabilidad esperada es 25.00%
"2" fue muestreado 6249 veces. Eso es 62.49%, y la probabilidad esperada es 62.50%


## Softmaxificación

In [12]:
# Aplicar softmax al vector
vectSoftmax = F.softmax(vect, dim=-1)

# Nuevo reporte
for v, c, vectval in zip(vals, counts, vectSoftmax):

  observedFrequency = c * 100 / len(mn)

  print(
      f'"{v}" fue muestreado {c:4} veces. Eso es {observedFrequency:5.2f}%, y la'
      f' probabilidad de softmax es {vectval*100:5.2f}%'
  )

"0" fue muestreado 1247 veces. Eso es 12.47%, y la probabilidad de softmax es  1.71%
"1" fue muestreado 2504 veces. Eso es 25.04%, y la probabilidad de softmax es  4.66%
"2" fue muestreado 6249 veces. Eso es 62.49%, y la probabilidad de softmax es 93.62%


Porque no coinciden? porque estamos tomando estos datos de vals y count que viene de donde estamos usando el vector original antes de softmax. Asi que esto es PROBABILIDAD DE SOFTMAX ESPERADA PERO NO PROBABILIDADES DE SOFTMAX OBSERVADAS.

In [ ]:
# Ahora si usando la transformacion de softmax

In [13]:
# Ahora muestreando a partir del vector softmax
mn = torch.multinomial(vectSoftmax, 10000, replacement=True)
vals, counts = np.unique(mn, return_counts=True)

# Nuevo reporte
for v, c, vectval in zip(vectSoftmax, counts, vectSoftmax):

  observedFrequency = c * 100 / len(mn)

  print(
      f'"{v:.4f}" fue muestreado {c:4} veces. Eso es {observedFrequency:5.2f}%, y la'
      f' probabilidad de softmax es {vectval*100:5.2f}%'
  )

"0.0171" fue muestreado  181 veces. Eso es  1.81%, y la probabilidad de softmax es  1.71%
"0.0466" fue muestreado  442 veces. Eso es  4.42%, y la probabilidad de softmax es  4.66%
"0.9362" fue muestreado 9377 veces. Eso es 93.77%, y la probabilidad de softmax es 93.62%


### Elección con Numpy

In [ ]:
np.random.choice(vect,10)

array([5., 2., 5., 1., 1., 1., 2., 5., 1., 1.], dtype=float32)

In [15]:
# Muestrear una gran cantidad de valores
mn = np.random.choice(vect, 10000, replace=True)

# Reportar los resultados
vals, counts = np.unique(mn, return_counts=True)
for v, c, vectval in zip(vals, counts, vect):

  observedFrequency = c * 100 / len(mn)
  expectedFrequency = 1 * 100 / len(vect)

  print(
      f'"{v}" fue muestreado {c} veces. Eso es {observedFrequency:.2f}%, y la'
      f' probabilidad esperada es {expectedFrequency:.2f}%'
  )

"1.0" fue muestreado 3428 veces. Eso es 34.28%, y la probabilidad esperada es 33.33%
"2.0" fue muestreado 3313 veces. Eso es 33.13%, y la probabilidad esperada es 33.33%
"5.0" fue muestreado 3259 veces. Eso es 32.59%, y la probabilidad esperada es 33.33%


In [16]:
### Hacer que np.random.choice coincida con la función de multinomial

# Definir probabilidades (pesos para la selección)
probvalues = vect / sum(vect)


# Muestrear una gran cantidad de valores
mn = np.random.choice(vect, 10000, replace=True, p=probvalues)

# Reportar los resultados
vals, counts = np.unique(mn, return_counts=True)
for v, c, p in zip(vals, counts, probvalues):

  observedFrequency = c * 100 / len(mn)
  expectedFrequency = p * 100

  print(
      f'"{v}" fue muestreado {c} veces. Eso es {observedFrequency:.2f}%, y la'
      f' probabilidad esperada es {expectedFrequency:.2f}%'
  )

"1.0" fue muestreado 1317 veces. Eso es 13.17%, y la probabilidad esperada es 12.50%
"2.0" fue muestreado 2428 veces. Eso es 24.28%, y la probabilidad esperada es 25.00%
"5.0" fue muestreado 6255 veces. Eso es 62.55%, y la probabilidad esperada es 62.50%


### **Análisis de `np.random.choice` y su Comparación con PyTorch**

En esta sección del notebook se explora cómo funciona el muestreo aleatorio en NumPy y qué debemos hacer para replicar el comportamiento de `torch.multinomial`.

---

### **1. Primera parte: Comportamiento por defecto (Muestreo Uniforme)**

```python
mn = np.random.choice(vect, 10000, replace=True)

```

* **Qué hace:** Selecciona elementos de `vect` al azar.
* **El "truco":** Por defecto, **NumPy no usa los valores del tensor como pesos**. Trata a los 3 elementos por igual, asignándoles una distribución uniforme:
* Probabilidad de elegir `1.0`: $1/3 \approx 33.33\%$
* Probabilidad de elegir `2.0`: $1/3 \approx 33.33\%$
* Probabilidad de elegir `5.0`: $1/3 \approx 33.33\%$


* **Resultado:** Al muestrear $10.000$ veces, cada número sale seleccionado aproximadamente el $33\%$ de las veces (~3.300 veces cada uno).

---

### **2. Segunda parte: Recreando `torch.multinomial` (Muestreo Ponderado)**

Para lograr que NumPy considere los valores de `vect` como probabilidades (como lo hace PyTorch), hay que hacer dos cosas:

1. **Normalizar los valores:** Dividir `vect` entre su suma para convertirlos en probabilidades reales que sumen $1.0$:

$$\text{probvalues} = \frac{[1, 2, 5]}{1 + 2 + 5} = [0.125, 0.250, 0.625]$$


2. **Pasar el parámetro `p`:**
```python
mn = np.random.choice(vect, 10000, replace=True, p=probvalues)

```



* **Resultado:** Ahora la probabilidad de cada número coincide con su peso original:
* `"1.0"` sale $\approx 13.00\%$ de las veces (esperado: $12.50\%$).
* `"2.0"` sale $\approx 25.03\%$ de las veces (esperado: $25.00\%$).
* `"5.0"` sale $\approx 61.97\%$ de las veces (esperado: $62.50\%$).



---

### **Dos diferencias clave entre `np.random.choice` y `torch.multinomial**`

| Característica | `np.random.choice(vect)` | `torch.multinomial(vect)` |
| --- | --- | --- |
| **¿Qué devuelve?** | Los **valores reales** (`1.0`, `2.0`, `5.0`). | Los **índices/posiciones** (`0`, `1`, `2`). |
| **Comportamiento por defecto** | Muestreo **uniforme** (misma probabilidad para todos), a menos que pases `p=...`. | Muestreo **ponderado** (usa los valores dentro de `vect` como pesos de probabilidad). |